# 01 · Data, physics, and feature hypotheses

**Question:** what can observed movement and supplied landing geometry tell us about motion after the throw?

Training exploration uses the frozen training partition. The supplied landing point and output horizon
are available at inference time; future player coordinates are targets, never features. The distributions
below come from the completed real-data benchmark, not generated examples.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
LOCAL = ROOT / "artifacts" / "benchmark"
PUBLISHED = ROOT / "docs" / "results"
RESULTS = LOCAL if (LOCAL / "summary.json").exists() else PUBLISHED
READY = (RESULTS / "summary.json").exists()
if READY:
    summary = json.loads((RESULTS / "summary.json").read_text())
    eda = json.loads((RESULTS / "eda.json").read_text())
    protocol = json.loads((RESULTS / "protocol.json").read_text())
    display(Markdown("**Report source:** " + ("local benchmark artifacts" if RESULTS == LOCAL else "published reproducible experiment snapshot")))
else:
    display(Markdown(
        "Run `nfl benchmark` after the data audit to create the real-data results. "
        "No synthetic result is substituted here."
    ))

## Freeze time before fitting

Keep all frames, players and plays of a game in one partition. These dates are checked against game identifiers and locked before training. Holdout labels are excluded from the development pipeline.

In [ ]:
if READY:
    display(pd.DataFrame.from_dict(protocol["partitions"], orient="index").loc[["train", "validation", "holdout"]])
    display(Markdown("**Holdout evaluation:** " + summary["holdout_evaluation"]))

## Data coverage

Input rows describe observed motion; target rows are the positions we score. A trajectory is one player within one play, and can contain several target frames.

In [ ]:
if READY:
    display(pd.DataFrame({"Training measure": ["Games", "Plays", "Scored trajectories", "Observed rows", "Target rows"], "Count": [eda[key] for key in ["games", "plays", "trajectories", "input_rows", "target_rows"]]}))
    display(pd.DataFrame(eda["weeks"])[["file", "games", "plays", "trajectories", "input_rows", "target_rows"]])

In [ ]:
if READY:
    display(Image(filename=str(RESULTS / "eda.png")))

## Football feature bank: measured hypotheses

The **2,843 candidates** cover actual frame lags, seven history windows, circular angles,
landing-relative dynamics, explicit receiver/passer anchors, nearest-player geometry, and forecast time.
Availability masks distinguish missing history and absent neighbors from observed zeros.
Player names, IDs as numerical predictors, and post-play outcomes are excluded.

The completed experiment used training-only screening and retained 64 columns per residual model.
A large feature count is a search space, not evidence that every feature helps.

In [ ]:
from nfl_trajectory.features import feature_catalog

catalog = feature_catalog()
display(Markdown(f"**Candidate feature count:** {len(catalog):,}"))
display(catalog.groupby("family").size().rename("Candidate features").to_frame())

In [ ]:
from nfl_trajectory.analysis import association_figure, figure_png, load_evidence

feature_summary, selection_audit, evidence_label = load_evidence(ROOT)
display(Markdown(f"**Feature experiment:** {evidence_label}."))
display(Image(data=figure_png(association_figure(feature_summary))))

### Interpret the relationships before choosing a larger model

`time_squared` has the largest recorded marginal association with the baseline residual.
Time-scaled lateral speed relative to the landing direction is next. Receiver-relative separation
and closest-approach timing also appear among strong training associations. This motivates
nonlinear time response and player-interaction hypotheses; it does **not** establish causal importance.

The reference role-ridge model already uses landing geometry. Consequently, **motion residual** means
motion-only *additional correction features*, not an end-to-end model denied the landing point.
All three challengers correct the same trained baseline. Screening uses its in-sample training
residuals; these are not out-of-fold feature-importance estimates.

In [ ]:
if selection_audit:
    shared = selection_audit["shared_landing_interaction"]
    removed = selection_audit["landing_not_in_interaction"]
    added = selection_audit["interaction_not_in_landing"]
    assert shared + len(removed) == 64
    assert shared + len(added) == 64
    display(pd.DataFrame({
        "Comparison": ["Shared columns", "Landing columns displaced", "Interaction columns added"],
        "Count": [shared, len(removed), len(added)],
    }))
    display(pd.DataFrame({"Displaced landing-model feature": removed}))
else:
    display(Markdown("No matching selection audit is available for this local experiment."))

### Controlled next experiment

Keep the winning landing model as a reference. Test a **protected landing feature block plus interaction
features**, with a capacity-matched noninteraction control. This distinguishes useful new information
from simply replacing useful columns. Receiver/defender role conditioning, relative velocity, closest
approach, and nonlinear time effects are the first hypotheses—not more anonymous feature crosses.

Choose screening, scaling, regularization, and model settings inside forward-chaining folds of the
training games. Refit the baseline inside each fold too: cached residuals from a baseline fitted to
all training games would contaminate those inner validation folds. Only then use the existing
32-game development partition for comparison. The later holdout remains untouched.

A later temporal attention challenger can represent observed history and surrounding players jointly.
The competition's [third-place writeup](https://www.kaggle.com/competitions/nfl-big-data-bowl-2026-prediction/writeups/3rd-place-solution)
reports a spatiotemporal Transformer with auxiliary losses. [Wayformer](https://waymo.com/research/wayformer/)
studies attention-based fusion for motion forecasting in driving. Those are external methodological
references, not evidence of an implemented NFL neural model here. Football-specific benefit must be
measured against the frozen protocol.